In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score

from xgboost import XGBClassifier

PROJECT_ROOT = Path().resolve()
while not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

from src.features import *
from src.volatility import *
from src.labeling import *

In [2]:
df = pd.read_csv("../data/raw/nifty50.csv", parse_dates=["Date"])
df = df.sort_values("Date").reset_index(drop=True)

df = add_returns(df)
df = add_close_to_close_volatility(df, window=20)

hmm_df = pd.read_csv("../data/processed/hmm_regimes.csv", parse_dates=["Date"])

df = df.merge(hmm_df, on="Date", how="inner")
df = df.dropna().reset_index(drop=True)

In [3]:
df = add_target_direction(df)
df = add_lagged_returns(df, lags=(1,5,10))
df = add_moving_average_features(df, windows=(5,10,20))
df = add_volatility_features(df, vol_col="vol_cc", lags=(1,))

df = df.dropna().reset_index(drop=True)

In [4]:
feature_cols = [
    "ret_lag_1","ret_lag_5","ret_lag_10",
    "ma_ratio_5","ma_ratio_10","ma_ratio_20",
    "vol_cc","vol_cc_lag_1"
]

In [5]:
split = int(len(df)*0.8)

train = df.iloc[:split].copy()
test  = df.iloc[split:].copy()

y_train = train["target"]
y_test  = test["target"]

In [6]:
scaler = StandardScaler()

X_train = scaler.fit_transform(train[feature_cols])
X_test  = scaler.transform(test[feature_cols])

# **Training Gradient Boosting**

In [7]:
gb = GradientBoostingClassifier(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    random_state=42
)

gb.fit(X_train, y_train)
pred_gb = gb.predict(X_test)

# **Training XGboost**

In [8]:
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

xgb.fit(X_train, y_train)
pred_xgb = xgb.predict(X_test)

In [9]:
global_results = pd.DataFrame({
    "Model": ["GradientBoost","XGBoost"],
    "Accuracy": [
        accuracy_score(y_test, pred_gb),
        accuracy_score(y_test, pred_xgb)
    ]
})

global_results

,Model,Accuracy
0,GradientBoost,0.556299
1,XGBoost,0.537347


In [10]:
test["pred_gb"] = pred_gb
test["pred_xgb"] = pred_xgb

def regime_acc(df, col):
    return df.groupby("hmm_regime").apply(
        lambda g: (g[col] == g["target"]).mean()
    )

regime_results = pd.DataFrame({
    "GB": regime_acc(test,"pred_gb"),
    "XGB": regime_acc(test,"pred_xgb")
})

regime_results

,GB,XGB
hmm_regime,,
High,0.659091,0.590909
Low,0.575117,0.549296
Medium,0.526932,0.519906


In [11]:
regime_results["XGB_gain"] = regime_results["XGB"] - regime_results["GB"]
regime_results

,GB,XGB,XGB_gain
hmm_regime,,,
High,0.659091,0.590909,-0.068182
Low,0.575117,0.549296,-0.025822
Medium,0.526932,0.519906,-0.007026
